# Import

In [ ]:
import pandas as pd

In [42]:
df = pd.read_excel(
    "/Users/wenye/SideProjects/Data Model/Stream-Assistant/Stream-Assistant/src/backend/analyze/Data Model - Vectorstore Eval.xlsx"
)
df

,Query,Answer
0,建構可擴展分散式系統的最佳工具有哪些？,"distributed, Distributed Locks, http, reverse-..."
1,心理學如何影響科技在社會中的採用？,"psychology, technology, society, lifestyle"
2,為行動開發學習 Kotlin Multiplatform 有哪些好處？,"kotlin-multiplatform, android, ios, education"
3,平面設計如何提升社群媒體上的用戶參與度？,"graphic-design, social-media, images, lifestyle"
4,軟體開發接案需要掌握哪些技能？,"freelancing, front end, freelance-writing, jav..."
5,深度學習如何促進大數據分析？,"deep-learning, Big Data, ai, python"
6,實作正向與反向代理時有哪些安全考量？,"reverse-proxy, forwarding-proxy, firewall, net..."
7,remarkable-tablets 在筆記與生產力方面有何特色？,"remarkable-tablet, notes, work, lifestyle"
8,遠距工作如何提升生產力與健康？,"work-from-home, work, health, self-improvement"
9,使用 Golang 建立基於 websocket 的應用程式有何優勢？,"golang, websocket, http, network"


In [14]:
from google.oauth2 import service_account
from google.cloud import bigquery

import src.backend.configs as configs

CREDENTIAL_OBJ = service_account.Credentials.from_service_account_file(
    filename="/Users/wenye/SideProjects/Data Model/Stream-Assistant/Stream-Assistant/final-project-vectorestore_credentials.json"
)
BQ_CLIENT = bigquery.Client(project=configs.PROJECT_ID, credentials=CREDENTIAL_OBJ)

# Utils

In [18]:
from datetime import datetime, timedelta


def get_time_range(base_time=None):
    """
    返回基於每天早上 7:00 計算的資料範圍。
    情境一：如果當前時間在今天 7:00 之前，返回前兩天 7:00 到前一天 7:00；
    情境二：如果當前時間在今天 7:00 之後，返回前一天 7:00 到今天 7:00。

    Args:
        base_time (datetime, optional): 指定的基準時間，默認為當前時間。

    Returns:
        tuple: (start_time, end_time) 分別表示資料的開始和結束時間。
    """
    if base_time is None:
        base_time = datetime.now()

    today_7am = base_time.replace(hour=7, minute=0, second=0, microsecond=0)

    if base_time < today_7am:  # 情境一
        start_time = today_7am - timedelta(days=2)
        end_time = today_7am - timedelta(days=1)
    else:  # 情境二
        start_time = today_7am - timedelta(days=1)
        end_time = today_7am

    print(f"Currrnt Time: {base_time}")
    print(f"Currrnt Time 7 am: {today_7am}")
    print(f"start_time: {start_time}")
    print(f"end_time: {end_time}")

    return start_time, end_time

# Get current tags in BQ

In [43]:
import sys
import os

# 將 `src` 的父目錄添加到 sys.path
project_root = "/Users/wenye/SideProjects/Data Model/Stream-Assistant/Stream-Assistant"
sys.path.append(project_root)

import src.backend.configs as configs

article_table_ref = f"{configs.PROJECT_ID}.{configs.DATASET_ID}.{configs.ARTICLE_INFO_TABLE_ID}"


# 獲取時間範圍
yesterday, today = get_time_range()

# 查詢
query = f"""
SELECT DISTINCT tags
FROM `{article_table_ref}`
WHERE publish_date BETWEEN TIMESTAMP('{yesterday}') AND TIMESTAMP('{today}')
"""

# 執行查詢
query_job = BQ_CLIENT.query(query)

# 獲取結果
distinct_tags = [tag for row in query_job.result() for tag in row.tags]

# 打印合并后的列表
print(distinct_tags)

print(f"set(distinct_tags): \n\n{set(distinct_tags)}")

Currrnt Time: 2024-12-30 20:14:33.603324
Currrnt Time 7 am: 2024-12-30 07:00:00
start_time: 2024-12-29 07:00:00
end_time: 2024-12-30 07:00:00
['airdrop', 'airdrops-minter', 'airdrops-tools', 'jupiter', 'jupiter-aggregator', 'jupiter-airdrop', 'airdrop-jupiter', 'jupiter-checker', 'jupiter-volume-checker', 'android', 'expo', 'ios', 'react', 'react-native', 'reanimated', 'scrollview', 'transition', 'atomicredteam', 'detection-engineering', 'sigma-rules', 'splunk', 'defaults', 'macos', 'vscode', 'bing-wallpaper', 'macos', 'wallpaper', 'ai', 'kotlin-multiplatform', 'ai', 'bolt', 'copilot', 'fictional', 'gemini-api', 'yemot', 'cursor-chinese-tutorial', 'cursor-examples', 'cursor-tutorial', 'commute', 'commuter', 'fare', 'philippines', 'puj', 'firewall', 'forwarding-proxy', 'golang', 'http', 'nat', 'proxy', 'reverse-proxy', 'websocket', 'remarkable-tablet', 'smtp', 'discord', 'image-generation', 'images', 'python', 'writing', 'education', 'life', 'society', 'work', 'education', 'work', 'life

檢查 df 和 bq 是否一樣

In [69]:
# 檢查是否所有主題都在 distinct_tags 中
missing_entries = []

for index, row in df.iterrows():
    query = row["Query"]
    answers = row["Answer"].split(", ")  # 分割成主題列表
    for topic in answers:
        if topic not in set(distinct_tags):
            missing_entries.append({"Query": query, "Missing Topic": topic})
            print(f"missing topic: {topic}")

# 將結果轉為 DataFrame 並顯示
missing_df = pd.DataFrame(missing_entries)
missing_df

""


# 向量查詢 function

In [61]:
from langchain_google_vertexai import VertexAIEmbeddings
from langchain_google_community import BigQueryVectorStore


import src.backend.configs as configs

today_date = today.strftime("%Y%m%d")
table_name = f"{configs.VECTORSTORE_TABLE_ID}_{today_date}"
print(f"table_name: {table_name}")


def get_vectorestore_obj():
    return BigQueryVectorStore(
        project_id=configs.PROJECT_ID,
        dataset_name=configs.DATASET_ID,
        table_name=table_name,
        location=configs.VECTORSTORE_REGION,
        embedding=VertexAIEmbeddings(
            model_name="textembedding-gecko@latest", project=configs.PROJECT_ID, credentials=CREDENTIAL_OBJ
        ),
        credentials=CREDENTIAL_OBJ,
        distance_type="COSINE",  # 'COSINE', 'EUCLIDEAN', 'DOT_PRODUCT'
    )


def search_similar_tags(query: list[str], sources: list[str] | None) -> list[(str)]:
    """Searches for similar tags based on the query.

    Args:
        query (str): The query to search for. e.g. "Give me some information about NLP."
        sources (list[str], optional): The sources selected by user to search. Defaults to None.
                                        e.g. ["github", "medium"]

    Returns:
        list[(str)]: A list of similar tags. e.g. ["NLP", "Natural Language Processing"]
    """
    vectorstore_obj = get_vectorestore_obj()

    interested_tags = []
    sources = sources if sources else ["github", "medium", "csdn"]

    filter = " OR ".join([f"source = '{source}'" for source in sources])

    docs_for_tags = vectorstore_obj.batch_search(
        queries=query,
        filter=f"({filter})",
        k=5,
    )

    for result in docs_for_tags:
        query_predictions = []
        for doc in result:
            query_predictions.append(doc[0].page_content)
        interested_tags.append(query_predictions)

    return interested_tags

table_name: doc_and_vectors_20241230


In [62]:
result = search_similar_tags(query=df["Answer"].tolist(), sources=None)
print(f"result: \n{result}")

BigQuery table final-project-1-444506.final_project_poc.doc_and_vectors_20241230 initialized/validated as persistent storage. Access via BigQuery console:
 https://console.cloud.google.com/bigquery?project=final-project-1-444506&ws=!1m5!1m4!4m3!1sfinal-project-1-444506!2sfinal_project_poc!3sdoc_and_vectors_20241230
result: 
[['Distributed Locks', 'reverse-proxy', 'distributed', 'http', 'forwarding-proxy'], ['psychology', 'lifestyle', 'technology', 'society', 'social-media'], ['kotlin-multiplatform', 'android', 'golang', 'multilingual', 'ios'], ['graphic-design', 'social-media', 'lifestyle', 'images', 'life'], ['freelancing', 'freelance-writing', 'javascript', 'front end', 'html'], ['deep-learning', 'Big Data', 'python', 'ai', 'machine-learning-algorithms'], ['reverse-proxy', 'forwarding-proxy', 'firewall', 'proxy', 'network'], ['remarkable-tablet', 'notebook', 'notes', 'note-taking', 'lifestyle'], ['work-from-home', 'self-improvement', 'health', 'freelancing', 'work'], ['websocket', 'g

In [63]:
df_test = df.copy()
df_test["Prediction"] = result
df_test

,Query,Answer,Prediction
0,建構可擴展分散式系統的最佳工具有哪些？,"distributed, Distributed Locks, http, reverse-...","[Distributed Locks, reverse-proxy, distributed..."
1,心理學如何影響科技在社會中的採用？,"psychology, technology, society, lifestyle","[psychology, lifestyle, technology, society, s..."
2,為行動開發學習 Kotlin Multiplatform 有哪些好處？,"kotlin-multiplatform, android, ios, education","[kotlin-multiplatform, android, golang, multil..."
3,平面設計如何提升社群媒體上的用戶參與度？,"graphic-design, social-media, images, lifestyle","[graphic-design, social-media, lifestyle, imag..."
4,軟體開發接案需要掌握哪些技能？,"freelancing, front end, freelance-writing, jav...","[freelancing, freelance-writing, javascript, f..."
5,深度學習如何促進大數據分析？,"deep-learning, Big Data, ai, python","[deep-learning, Big Data, python, ai, machine-..."
6,實作正向與反向代理時有哪些安全考量？,"reverse-proxy, forwarding-proxy, firewall, net...","[reverse-proxy, forwarding-proxy, firewall, pr..."
7,remarkable-tablets 在筆記與生產力方面有何特色？,"remarkable-tablet, notes, work, lifestyle","[remarkable-tablet, notebook, notes, note-taki..."
8,遠距工作如何提升生產力與健康？,"work-from-home, work, health, self-improvement","[work-from-home, self-improvement, health, fre..."
9,使用 Golang 建立基於 websocket 的應用程式有何優勢？,"golang, websocket, http, network","[websocket, golang, javascript, http, webapp]"


In [66]:
# 計算每行的 Precision, Recall 和 F1 Score
results = []
for index, row in df_test.iterrows():
    # 將 Answer 分割為集合
    answer_set = set(row["Answer"].split(", "))
    prediction_set = set(row["Prediction"])

    # 計算 True Positives, False Positives, 和 False Negatives
    tp = len(answer_set & prediction_set)  # 正確預測的元素
    fp = len(prediction_set - answer_set)  # 錯誤預測的元素
    fn = len(answer_set - prediction_set)  # 遺漏的正確元素

    # 計算 Precision 和 Recall
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0

    # 計算 F1 Score
    f1_score = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

    # 保存結果
    results.append(
        {
            "Query": row["Query"],
            "Precision": precision,
            "Recall": recall,
            "F1 Score": f1_score,
        }
    )

# 將結果轉為 DataFrame 並顯示
result_df = pd.DataFrame(results)
result_df

,Query,Precision,Recall,F1 Score
0,建構可擴展分散式系統的最佳工具有哪些？,0.8,1.00,0.888889
1,心理學如何影響科技在社會中的採用？,0.8,1.00,0.888889
2,為行動開發學習 Kotlin Multiplatform 有哪些好處？,0.6,0.75,0.666667
3,平面設計如何提升社群媒體上的用戶參與度？,0.8,1.00,0.888889
4,軟體開發接案需要掌握哪些技能？,0.8,1.00,0.888889
5,深度學習如何促進大數據分析？,0.8,1.00,0.888889
6,實作正向與反向代理時有哪些安全考量？,0.8,1.00,0.888889
7,remarkable-tablets 在筆記與生產力方面有何特色？,0.6,0.75,0.666667
8,遠距工作如何提升生產力與健康？,0.8,1.00,0.888889
9,使用 Golang 建立基於 websocket 的應用程式有何優勢？,0.6,0.75,0.666667


In [67]:
macro_precision = sum(result["Precision"] for result in results) / len(results)
macro_recall = sum(result["Recall"] for result in results) / len(results)
macro_f1 = sum(result["F1 Score"] for result in results) / len(results)

print("Macro-Averaged Results:")
print(f"Precision: {macro_precision:.4f}")
print(f"Recall: {macro_recall:.4f}")
print(f"F1 Score: {macro_f1:.4f}")

Macro-Averaged Results:
Precision: 0.7200
Recall: 0.9125
F1 Score: 0.8042


# 加入 represent

In [84]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.output_parsers import ResponseSchema, StructuredOutputParser
from langchain_core.prompts import PromptTemplate

from langchain_google_vertexai import VertexAIEmbeddings
from langchain_google_community import BigQueryVectorStore

import src.backend.configs as configs
from src.backend.configs import GOOGLE_API_KEY
import src.backend.configs as configs

today_date = today.strftime("%Y%m%d")
table_name = f"{configs.VECTORSTORE_TABLE_ID}_{today_date}"
print(f"table_name: {table_name}")


def get_vectorestore_obj():
    return BigQueryVectorStore(
        project_id=configs.PROJECT_ID,
        dataset_name=configs.DATASET_ID,
        table_name=table_name,
        location=configs.VECTORSTORE_REGION,
        embedding=VertexAIEmbeddings(
            model_name="textembedding-gecko@latest", project=configs.PROJECT_ID, credentials=CREDENTIAL_OBJ
        ),
        credentials=CREDENTIAL_OBJ,
        distance_type="COSINE",  # 'COSINE', 'EUCLIDEAN', 'DOT_PRODUCT'
    )


def represent_query(query: str) -> list[str]:

    llm = ChatGoogleGenerativeAI(
        model="gemini-1.5-flash-8b",
        temperature=0,
        max_tokens=None,
        timeout=None,
        max_retries=2,
        credentials=CREDENTIAL_OBJ,
        api_key=GOOGLE_API_KEY,  # type: ignore
    )

    tags_parser = StructuredOutputParser.from_response_schemas(
        response_schemas=[
            ResponseSchema(
                name="tags",
                description=(
                    "Provide a list of topics in English that accurately represent the key themes or subjects "
                    "discussed in the query. The topics should be concise, relevant, and capture the essence of "
                    "what the query is about."
                ),
                type="List[str]",
            ),
        ]
    )
    prompt_template = """
    You are an intelligent assistant that helps identify key topics or tags from natural language queries. Your task is to analyze the user's input and provide a concise, relevant list of English tags that represent the main themes or subjects mentioned or implied in the query.

    Guidelines:
    1. Focus on extracting key topics that best describe the content of the query.
    2. Use clear and specific tags that are relevant to the query's context.
    3. Avoid using vague or overly broad tags; ensure each tag is meaningful.
    4. Return the tags as a list of strings in English.
    5. you have to follow the format to response: {format_instructions}

    Example 1:
    Query: "What are the best tools for building scalable distributed systems?"
    Tags: ["distributed", "Distributed Locks", "http", "reverse-proxy"]

    Example 2:
    Query: "How does psychology influence technology adoption in society?"
    Tags: ["psychology", "technology", "society", "lifestyle"]

    Now, analyze the following query and provide the relevant tags:
    {task_content}
    """

    chat_prompt = PromptTemplate(
        input_variables=["task_content"],
        partial_variables={
            "format_instructions": tags_parser.get_format_instructions(),
        },
        template=prompt_template,
        output_parser=tags_parser,
    )

    chain = chat_prompt | llm | tags_parser
    response = chain.invoke({"task_content": query})

    print(f"response: {response}")

    return response["tags"]


def search_similar_tags_with_represent(queries: list[str], sources: list[str] | None) -> list[(str)]:
    """Searches for similar tags based on the query.

    Args:
        query (str): The query to search for. e.g. "Give me some information about NLP."
        sources (list[str], optional): The sources selected by user to search. Defaults to None.
                                        e.g. ["github", "medium"]

    Returns:
        list[(str)]: A list of similar tags. e.g. ["NLP", "Natural Language Processing"]
    """
    vectorstore_obj = get_vectorestore_obj()

    interested_tags = []
    sources = sources if sources else ["github", "medium", "csdn"]

    filter = " OR ".join([f"source = '{source}'" for source in sources])

    top_5_tags_result = []
    for query in queries:
        represented_query = represent_query(query)
        print(f"represented_query: {represented_query}")

        docs_for_tags = vectorstore_obj.batch_search(
            queries=represented_query,
            filter=f"({filter})",
            k=5,
        )

        interested_tags = []
        for result in docs_for_tags:
            query_predictions = []
            for doc in result:
                query_predictions.append({"content": doc[0].page_content, "similarity": doc[1]})
            interested_tags.extend(query_predictions)

        top_5_tags_with_score = sorted(interested_tags, key=lambda x: x["similarity"], reverse=True)[:5]
        top_5_tags = [tag["content"] for tag in top_5_tags_with_score]

        print(f"top_5_tags: {top_5_tags}")

        top_5_tags_result.append(top_5_tags)

    return top_5_tags_result

table_name: doc_and_vectors_20241230


In [85]:
df_test_2 = df.copy()

result = search_similar_tags_with_represent(queries=df_test_2["Answer"].tolist(), sources=None)
print(f"result: \n{result}")

df_test_2["Prediction"] = result
df_test_2

BigQuery table final-project-1-444506.final_project_poc.doc_and_vectors_20241230 initialized/validated as persistent storage. Access via BigQuery console:
 https://console.cloud.google.com/bigquery?project=final-project-1-444506&ws=!1m5!1m4!4m3!1sfinal-project-1-444506!2sfinal_project_poc!3sdoc_and_vectors_20241230
response: {'tags': ['distributed', 'Distributed Locks', 'http', 'reverse-proxy']}
represented_query: ['distributed', 'Distributed Locks', 'http', 'reverse-proxy']
interested_tags: [{'content': 'distributed', 'similarity': 1.920498561003292e-05}, {'content': 'Distributed Locks', 'similarity': 0.20315066532787118}, {'content': 'public', 'similarity': 0.29766016824222596}, {'content': 'network', 'similarity': 0.3160118664909608}, {'content': 'streaming', 'similarity': 0.31799623452942116}, {'content': 'Distributed Locks', 'similarity': 2.7467059869001886e-05}, {'content': 'distributed', 'similarity': 0.20322030117736334}, {'content': 'security', 'similarity': 0.3504134365311536

Retrying langchain_google_genai.chat_models._chat_with_retry.<locals>._chat_with_retry in 2.0 seconds as it raised PermissionDenied: 403 Generative Language API has not been used in project 957230402857 before or it is disabled. Enable it by visiting https://console.developers.google.com/apis/api/generativelanguage.googleapis.com/overview?project=957230402857 then retry. If you enabled this API recently, wait a few minutes for the action to propagate to our systems and retry. [links {
  description: "Google developers console API activation"
  url: "https://console.developers.google.com/apis/api/generativelanguage.googleapis.com/overview?project=957230402857"
}
, reason: "SERVICE_DISABLED"
domain: "googleapis.com"
metadata {
  key: "service"
  value: "generativelanguage.googleapis.com"
}
metadata {
  key: "consumer"
  value: "projects/957230402857"
}
].


response: {'tags': ['tablet', 'notes', 'work', 'lifestyle']}
represented_query: ['tablet', 'notes', 'work', 'lifestyle']
interested_tags: [{'content': 'remarkable-tablet', 'similarity': 0.22627691267810301}, {'content': 'android', 'similarity': 0.25964170197274505}, {'content': 'pdf', 'similarity': 0.2614801512295414}, {'content': 'notebook', 'similarity': 0.27436463305131287}, {'content': 'books', 'similarity': 0.28237881383371466}, {'content': 'notes', 'similarity': 1.8431403381424794e-05}, {'content': 'note-taking', 'similarity': 0.11754511986543603}, {'content': 'notes-app', 'similarity': 0.1372058831862748}, {'content': 'notebook', 'similarity': 0.1897382812799916}, {'content': 'evernote', 'similarity': 0.22764379446892358}, {'content': 'work', 'similarity': 9.43304814116086e-06}, {'content': 'work-from-home', 'similarity': 0.24605829904354193}, {'content': 'writing', 'similarity': 0.25562127432136483}, {'content': 'life', 'similarity': 0.26656542834000696}, {'content': 'twitter',

Retrying langchain_google_genai.chat_models._chat_with_retry.<locals>._chat_with_retry in 2.0 seconds as it raised PermissionDenied: 403 Generative Language API has not been used in project 957230402857 before or it is disabled. Enable it by visiting https://console.developers.google.com/apis/api/generativelanguage.googleapis.com/overview?project=957230402857 then retry. If you enabled this API recently, wait a few minutes for the action to propagate to our systems and retry. [links {
  description: "Google developers console API activation"
  url: "https://console.developers.google.com/apis/api/generativelanguage.googleapis.com/overview?project=957230402857"
}
, reason: "SERVICE_DISABLED"
domain: "googleapis.com"
metadata {
  key: "service"
  value: "generativelanguage.googleapis.com"
}
metadata {
  key: "consumer"
  value: "projects/957230402857"
}
].


response: {'tags': ['golang', 'websocket', 'http', 'network', 'programming', 'websockets']}
represented_query: ['golang', 'websocket', 'http', 'network', 'programming', 'websockets']
interested_tags: [{'content': 'golang', 'similarity': 2.7735270476170193e-05}, {'content': 'javascript', 'similarity': 0.260053208896614}, {'content': 'python', 'similarity': 0.26036595984066935}, {'content': 'english', 'similarity': 0.27305705360315713}, {'content': 'github', 'similarity': 0.27717609386548214}, {'content': 'websocket', 'similarity': 2.1577953600138144e-05}, {'content': 'webapp', 'similarity': 0.20131696438834457}, {'content': 'javascript', 'similarity': 0.25181738365761175}, {'content': 'webdav', 'similarity': 0.26798260796056805}, {'content': 'http', 'similarity': 0.2821244641762354}, {'content': 'http', 'similarity': 2.6682900845398372e-05}, {'content': 'html', 'similarity': 0.1738564816205429}, {'content': 'api', 'similarity': 0.24830813206692692}, {'content': 'github', 'similarity': 0

,Query,Answer,Prediction
0,建構可擴展分散式系統的最佳工具有哪些？,"distributed, Distributed Locks, http, reverse-...","[windows, hacks, security, http, search-engines]"
1,心理學如何影響科技在社會中的採用？,"psychology, technology, society, lifestyle","[AI, society, lifestyle, theme, health]"
2,為行動開發學習 Kotlin Multiplatform 有哪些好處？,"kotlin-multiplatform, android, ios, education","[github, react-native, golang, webapp, react-n..."
3,平面設計如何提升社群媒體上的用戶參與度？,"graphic-design, social-media, images, lifestyle","[society, theme, data-analytics, images, society]"
4,軟體開發接案需要掌握哪些技能？,"freelancing, front end, freelance-writing, jav...","[html, graphic-design, javascript, react-nativ..."
5,深度學習如何促進大數據分析？,"deep-learning, Big Data, ai, python","[Internet of Things, github, language, linux, ..."
6,實作正向與反向代理時有哪些安全考量？,"reverse-proxy, forwarding-proxy, firewall, net...","[prompt-engineering, nextjs, http, search-engi..."
7,remarkable-tablets 在筆記與生產力方面有何特色？,"remarkable-tablet, notes, work, lifestyle","[books, society, theme, notebook, twitter]"
8,遠距工作如何提升生產力與健康？,"work-from-home, work, health, self-improvement","[psychology, life-lessons, development, educat..."
9,使用 Golang 建立基於 websocket 的應用程式有何優勢？,"golang, websocket, http, network","[http, firewall, http, social-media, github]"


In [86]:
# 計算每行的 Precision, Recall 和 F1 Score
results = []
for index, row in df_test_2.iterrows():
    # 將 Answer 分割為集合
    answer_set = set(row["Answer"].split(", "))
    prediction_set = set(row["Prediction"])

    # 計算 True Positives, False Positives, 和 False Negatives
    tp = len(answer_set & prediction_set)  # 正確預測的元素
    fp = len(prediction_set - answer_set)  # 錯誤預測的元素
    fn = len(answer_set - prediction_set)  # 遺漏的正確元素

    # 計算 Precision 和 Recall
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0

    # 計算 F1 Score
    f1_score = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

    # 保存結果
    results.append(
        {
            "Query": row["Query"],
            "Precision": precision,
            "Recall": recall,
            "F1 Score": f1_score,
        }
    )

# 將結果轉為 DataFrame 並顯示
result_df_with_represent = pd.DataFrame(results)
result_df_with_represent

,Query,Precision,Recall,F1 Score
0,建構可擴展分散式系統的最佳工具有哪些？,0.20,0.250000,0.222222
1,心理學如何影響科技在社會中的採用？,0.40,0.500000,0.444444
2,為行動開發學習 Kotlin Multiplatform 有哪些好處？,0.00,0.000000,0.000000
3,平面設計如何提升社群媒體上的用戶參與度？,0.25,0.250000,0.250000
4,軟體開發接案需要掌握哪些技能？,0.20,0.250000,0.222222
5,深度學習如何促進大數據分析？,0.00,0.000000,0.000000
6,實作正向與反向代理時有哪些安全考量？,0.00,0.000000,0.000000
7,remarkable-tablets 在筆記與生產力方面有何特色？,0.00,0.000000,0.000000
8,遠距工作如何提升生產力與健康？,0.00,0.000000,0.000000
9,使用 Golang 建立基於 websocket 的應用程式有何優勢？,0.25,0.250000,0.250000


In [88]:
macro_precision = sum(result["Precision"] for result in results) / len(results)
macro_recall = sum(result["Recall"] for result in results) / len(results)
macro_f1 = sum(result["F1 Score"] for result in results) / len(results)

print("Macro-Averaged Results:")
print(f"Precision: {macro_precision:.4f}")
print(f"Recall: {macro_recall:.4f}")
print(f"F1 Score: {macro_f1:.4f}")

Macro-Averaged Results:
Precision: 0.1250
Recall: 0.1542
F1 Score: 0.1375
